In [ ]:
from plotting_celltypes_new import (get_animal_clean_dict_activity, fit_GLM_population, get_residual_activity_dict)
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import slicetca

In [6]:

filepath = '/Users/michaelfinch/CA1-interneuron-GLM/datasets/NDNF_E0A1B1_251107.mat'

animal_clean_dict_activity, animal_vel_dict, animal_trials_original, animal_trials_clean, trials_to_remove_local, animal_lick_dict = get_animal_clean_dict_activity(filepath)

GLM_params, predicted_activity_dict = fit_GLM_population(animal_vel_dict, animal_clean_dict_activity, quintile=None, regression='ridge', alphas=None)

residual_activity_dict_NDNF_new = get_residual_activity_dict(animal_clean_dict_activity, predicted_activity_dict)


clean_resid_activity_dict_NDNF_newest = {}

clean_velocity_dict_NDNF_newest = {}

clean_lick_dict_NDNF_newest = {}

for idx, animal in enumerate(residual_activity_dict_NDNF_new):
    if 14 < idx < 29:
        clean_resid_activity_dict_NDNF_newest[f"animal_{idx+1}"] = residual_activity_dict_NDNF_new[animal]
        clean_velocity_dict_NDNF_newest[f"animal_{idx+1}"] = animal_vel_dict[animal]
        clean_lick_dict_NDNF_newest [f"animal_{idx+1}"] = animal_lick_dict[animal]


# with open('/Users/michaelfinch/CA1-interneuron-GLM/datasets/NDNF_fixed_model_dict_clean.pkl', 'rb') as f:
#     NDNF_model_dict_clean  = pickle.load(f)


# save_path = '/Users/michaelfinch/CA1-interneuron-GLM/datasets/all_cells_truncated_fixed_model.pkl'
# with open(save_path, 'rb') as f:
#     sliceTCA_model = pickle.load(f)

In [31]:
tensor_per_animal_list = []

for animal in clean_resid_activity_dict_NDNF_newest:
    cell_list = []
    for cell in clean_resid_activity_dict_NDNF_newest[animal]:
        cell_list.append(clean_resid_activity_dict_NDNF_newest[animal][cell])

    cells_array = np.array(cell_list)
    cells_array = cells_array.transpose(2, 0, 1)
    tensor_per_animal_list.append(cells_array)

In [35]:
tensor_per_animal_list[2].type

AttributeError: 'numpy.ndarray' object has no attribute 'type'

In [43]:
per_num_latents_dict = {}

for i in range(1, 61):

    components = (0,i,0)

    model_per_animal_list = []

    for animal in range(len(tensor_per_animal_list)):

        cells_array = tensor_per_animal_list[animal]

        example_animal_tensor = torch.from_numpy(cells_array)

        components20, model_20 = slicetca.decompose(example_animal_tensor,
                                                    number_components=components, # (trials, neurons, time bins)
                                                    positive=False,learning_rate=1*10**-2, min_std=10**-5, max_iter=4000, iter_std=1000,seed=0)
        
        model_per_animal_list.append(model_20)

    per_num_latents_dict[i] = model_per_animal_list



Loss: 0.0643542374238373 : 100%|██████████| 4000/4000 [06:21<00:00, 10.50it/s] 
The model converged. Loss: 5.9097964357144075e-06 :  38%|███▊      | 1507/4000 [02:01<03:21, 12.38it/s]
Loss: 0.11601655943041746 : 100%|██████████| 4000/4000 [08:36<00:00,  7.74it/s]
The model converged. Loss: 4.4230973068681395e-06 :  40%|████      | 1610/4000 [9:58:53<14:49:02, 22.32s/it]
The model converged. Loss: 8.737270403560924e-06 :  40%|████      | 1615/4000 [01:50<02:43, 14.59it/s]
Loss: 0.08314147948458338 : 100%|██████████| 4000/4000 [06:08<00:00, 10.85it/s]
The model converged. Loss: 1.6962073728911978e-05 :  39%|███▉      | 1564/4000 [02:15<03:30, 11.58it/s]
The model converged. Loss: 3.771416790072206e-05 :  39%|███▉      | 1569/4000 [02:17<03:33, 11.39it/s]
Loss: 0.25386217444205 :  54%|█████▍    | 2151/4000 [04:55<04:13,  7.29it/s]   


KeyboardInterrupt: 

In [22]:
raw_model = model_per_animal_list[0]

In [ ]:
save_path = "./per_num_latents_dict_x00_ndnf.pkl"

with open(save_path, 'wb') as f:
    pickle.dump(per_num_latents_dict, f)
    print(f"saved to {save_path}")

In [28]:
reconstruction_full = raw_model.construct().numpy(force=True)
reconstruction_full.shape

tensor_per_animal_list[0].shape

MSE = np.mean(np.square(tensor_per_animal_list[0] - reconstruction_full))
print(MSE)

0.0002906931726205356


In [ ]:
for animal in range(len(tensor_per_animal_list)):

        cells_array = tensor_per_animal_list[animal]

        example_animal_tensor = torch.from_numpy(cells_array)

        print(example_animal_tensor.shape)

TypeError: 'NoneType' object is not callable